In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rockyt07/uber-sql-database/rideshare.db
/kaggle/input/datasets/rockyt07/uber-sql-database/schema.sql


In [7]:
import pandas as pd
import sqlite3
import os

# Try this path instead
for file in os.listdir("/kaggle/input"):
    print(file)

datasets


In [8]:
for file in os.listdir("/kaggle/input/datasets"):
    print(file)

rockyt07


In [9]:
for file in os.listdir("/kaggle/input/datasets/rockyt07"):
    print(file)

uber-sql-database


In [10]:
for file in os.listdir("/kaggle/input/datasets/rockyt07/uber-sql-database"):
    print(file)

rideshare.db
schema.sql


In [11]:
import sqlite3
import pandas as pd

# Connect directly to the database
conn = sqlite3.connect("/kaggle/input/datasets/rockyt07/uber-sql-database/rideshare.db")

# See all the tables inside it
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables)

            name
0          users
1        drivers
2         riders
3      locations
4          trips
5       payments
6        reviews
7  cancellations


In [12]:
# Preview each table
for table in ["users", "drivers", "riders", "locations", "trips", "payments", "reviews", "cancellations"]:
    print(f"\n--- {table.upper()} ---")
    df = pd.read_sql(f"SELECT * FROM {table} LIMIT 5", conn)
    print(df)


--- USERS ---
   user_id              name                        email            phone  \
0        1       David White        david.white@gmail.com  +1-809-169-4853   
1        2      Justin Young      justin.young@icloud.com  +1-808-140-2343   
2        3       Scott Kelly       scott.kelly@icloud.com  +1-523-367-4346   
3        4  Dorothy Phillips   dorothy.phillips@yahoo.com  +1-887-760-5915   
4        5   Barbara Johnson  barbara.johnson@hotmail.com  +1-836-676-2638   

          city date_joined  is_driver  
0      Houston  2019-09-03          0  
1      Houston  2022-09-08          1  
2  Los Angeles  2020-05-03          0  
3      Houston  2020-10-09          0  
4     New York  2022-01-06          1  

--- DRIVERS ---
   driver_id  user_id vehicle_make vehicle_model  vehicle_year license_plate  \
0          1     1781          Kia          Soul          2015      ZVJ-2140   
1          2      408      Hyundai        Tucson          2016      WLW-3906   
2          3      6

In [13]:
for table in ["users", "drivers", "riders", "locations", "trips", "payments", "reviews", "cancellations"]:
    print(f"\n--- {table.upper()} ---")
    df = pd.read_sql(f"SELECT * FROM {table} LIMIT 1", conn)
    print(df.columns.tolist())


--- USERS ---
['user_id', 'name', 'email', 'phone', 'city', 'date_joined', 'is_driver']

--- DRIVERS ---
['driver_id', 'user_id', 'vehicle_make', 'vehicle_model', 'vehicle_year', 'license_plate', 'rating', 'join_date', 'is_active']

--- RIDERS ---
['rider_id', 'user_id', 'rating', 'total_trips', 'created_at']

--- LOCATIONS ---
['location_id', 'zone_name', 'city', 'latitude', 'longitude', 'zone_type']

--- TRIPS ---
['trip_id', 'rider_id', 'driver_id', 'pickup_location_id', 'dropoff_location_id', 'requested_at', 'started_at', 'completed_at', 'status', 'distance_km', 'duration_mins', 'base_fare', 'surge_multiplier', 'total_fare', 'payment_method']

--- PAYMENTS ---
['payment_id', 'trip_id', 'amount', 'method', 'status', 'paid_at']

--- REVIEWS ---
['review_id', 'trip_id', 'reviewer_id', 'reviewee_id', 'rating', 'comment', 'reviewed_at']

--- CANCELLATIONS ---
['cancel_id', 'trip_id', 'cancelled_by', 'reason', 'cancelled_at']


In [14]:
query = """
SELECT city, COUNT(*) as total_users
FROM users
GROUP BY city
ORDER BY total_users DESC
"""

pd.read_sql(query, conn)

,city,total_users
0,Houston,552
1,Los Angeles,497
2,New York,480
3,Chicago,471


In [15]:
query = """
SELECT u.name, d.rating, d.vehicle_make, d.vehicle_model, d.vehicle_year
FROM drivers d
JOIN users u ON d.user_id = u.user_id
ORDER BY d.rating DESC
LIMIT 5
"""

pd.read_sql(query, conn)

,name,rating,vehicle_make,vehicle_model,vehicle_year
0,Cynthia Cook,5.00,Chevrolet,Silverado,2022
1,Patricia Bailey,5.00,Mazda,MX-5,2013
2,Brandon Reyes,4.99,Honda,CR-V,2014
3,Gregory Morris,4.99,Ford,Escape,2017
4,Donna Morales,4.99,Tesla,Model X,2015


In [16]:
query = """
SELECT 
    ROUND(AVG(total_fare), 2) as avg_fare,
    ROUND(MIN(total_fare), 2) as cheapest_trip,
    ROUND(MAX(total_fare), 2) as most_expensive_trip
FROM trips
"""

pd.read_sql(query, conn)

,avg_fare,cheapest_trip,most_expensive_trip
0,36.01,3.65,224.27


In [17]:
query = """
SELECT reason, COUNT(*) as total_cancellations
FROM cancellations
GROUP BY reason
ORDER BY total_cancellations DESC
"""

pd.read_sql(query, conn)

,reason,total_cancellations
0,personal emergency,396
1,too long wait,284
2,changed my mind,283
3,waited too long,262
4,price too high,250
5,duplicate booking,249
6,found another ride,248
7,driver too far,231
8,wrong pickup,137
9,system error,133


In [18]:
query = """
SELECT method, COUNT(*) as total_payments,
    ROUND(SUM(amount), 2) as total_revenue
FROM payments
GROUP BY method
ORDER BY total_payments DESC
"""

pd.read_sql(query, conn)

,method,total_payments,total_revenue
0,card,5700,208034.67
1,wallet,5621,200500.16
2,cash,5506,196055.09
